<a href="https://colab.research.google.com/github/kenzoyanome/brazilian_ecommerce/blob/main/bra_ecommerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olist
Identificar oportunidades de mejora en el negocio

### Bases de datos
customers_dataset:
  - customer_id
  - customer_unique_id
  - customer_zip_code_prefix
  - customer_city
  - customer_state

geolocation_dataset (**divided in 3 files**):
  - geolocation_zip_code_prefix
  - geolocation_lat
  - geolocation_lng
  - geolocation_city
  - geolocation_state

order_items_dataset:
  - order_id
  - order_item_id
  - product_id
  - seller_id
  - shipping_limit_date
  - price
  - freight_value

order_payments_dataset:
  - order_id
  - payment_sequential
  - payment_type
  - payment_installments
  - payment_value

order_reviews_dataset:  
  - review_id
  - order_id
  - review_score
  - review_comment_title
  - review_comment_message
  - review_creation_date
  - review_answer_timestamp

orders_dataset:
  - order_id
  - customer_id
  - order_status
  - order_purchase_timestamp
  - order_approved_at
  - order_delivered_carrier_date
  - order_delivered_customer_date
  - order_estimated_delivery_date

product_category_name_translation:
  - product_category_name
  - product_category_name_english

products_dataset:
  - product_id
  - product_category_name
  - product_name_lenght
  - product_description_lenght
  - product_photos_qty
  - product_weight_g
  - product_length_cm
  - product_height_cm
  - product_width_cm

sellers_dataset:
  - seller_id
  - seller_zip_code_prefix
  - seller_city
  - seller_state

### Preguntas de negocio
Ventas  
¿Cuánto vende Olist?  
¿Cómo evolucionan las ventas mensualmente?  
¿Cuál es el ticket promedio?  
¿Qué estados generan más ingresos?  

Clientes  
¿Quiénes son los clientes más valiosos?  
¿Cuántas compras realiza cada cliente?  
¿Hay señales de abandono?  

Productos  
¿Qué categorías generan más ingresos?  
¿Cuáles tienen mejores reseñas?  
¿Existe relación entre precio y satisfacción?  

Logística  
¿Cuánto tarda Olist en entregar?  
¿Qué porcentaje de pedidos llega tarde?  
¿La entrega tardía afecta la calificación?  

Vendedores  
¿Qué vendedores generan más ingresos?  
¿Cuáles tienen mejores calificaciones?  
¿Qué vendedores tienen problemas de entrega?  

---
Fase 1 - Python: limpieza, auditoría de calidad de datos, feature engineering (tiempos de entrega, retrasos, valor de pedido, etc.) y EDA con seaborn/matplotlib.

Fase 2 - SQL (PostgreSQL): funnel de conversión, cohortes, análisis de retención y rentabilidad usando window functions.

Fase 3 - Power BI: modelo de datos, medidas DAX, y diseño de 2-3 dashboards (Ejecutivo, Logística, Clientes).

---

### SQL
JOIN  
GROUP BY  
CASE WHEN  
CTE  
funciones de fecha  
subconsultas  
WINDOW FUNCTIONS  
COALESCE  
NULLIF  
agregaciones  

### Python
limpieza  
EDA  
detección de outliers  
análisis de distribución  
correlaciones  
análisis estadístico  
visualizaciones  
si encontramos una pregunta interesante, podemos hacer incluso una prueba estadística para comprobar si un hallazgo es realmente significativo.

### Power BI
Finalmente construiremos un dashboard con varias páginas:  
1. Executive Overview  
Ventas | órdenes | clientes | ticket promedio | entregas  
2. Sales & Products  
Categorías | productos | estados | evolución temporal  
3. Customer Analytics  
Clientes | RFM | frecuencia | valor monetario  
4. Logistics  
Tiempo de entrega | retrasos | estados | vendedores  
5. Customer Satisfaction  
Reviews | rating | retrasos vs satisfacción  

### Conclusiones
Deberiamos de termionar con un proyecto en GitHub, algo como:  
"Analicé aproximadamente 100,000 órdenes de un marketplace brasileño, integrando información de clientes, productos, vendedores, pagos, logística y satisfacción. Utilicé SQL para transformar y relacionar las fuentes, Python para el análisis exploratorio y Power BI para desarrollar un dashboard ejecutivo. Identifiqué los principales factores asociados con ventas, retrasos logísticos y satisfacción del cliente, y generé recomendaciones accionables."

In [1]:
# IMPORTAR LIBRERÍAS RELEVANTES
import numpy             as np
import pandas            as pd
import matplotlib.pyplot as plt
import seaborn           as sns

from scipy.stats                  import pointbiserialr, chi2_contingency, ttest_ind, levene, pointbiserialr, chi2_contingency, ttest_ind, ttest_1samp, mannwhitneyu, shapiro
from statsmodels.stats.proportion import proportions_ztest
# OTRAS CONFIGURACIONES: EN GENERAL, EN BLOQUE DE IMPORTS, SUELE EXISTIR UN ESPACIO DONDE SE HACEN DEFINICIONES GENERALES
#                       COMO ELIMINAR CIERTOS "WARNINGS" MOLESTOS, QUITAR LÍMITES DE PANDAS SOBRE CUANTAS FILAS O COLUMNAS
#                       MUESTRA AL MOMENTO DE IMPIRMIR EL DATAFRAME, DEFINIR ESTILOS PRE-DEFINIDOS PARA GRÁFICOS, ETC.
import warnings                                                  # MANEJO DE WARNINGS - ADVERTENCIAS
warnings.filterwarnings('ignore', category=FutureWarning)        # IGNORAR WARNINGS MOLESTOS
pd.set_option('display.max_columns', None)                       # ELIMINA LIMITES DE PANDAS PARA MOSTRAR COLUMNAS
pd.set_option('display.max_rows', None)                          # ELIMINA LIMITES DE PANDAS PARA MOSTRAR FILAS
pd.set_option('display.max_colwidth', None)                      # AUTOAJUSTA ANCHO DE COLUMNAS
pd.set_option('display.float_format', lambda x: '%.3f' % x)      # PERMITE EVADIR EL MOSTRAR NÚMEROS CON NOTACIÓN CINETÍFICA
plt.rcParams['figure.dpi'] = 140                                 # NIVEL DE RESOLUCIÓN.

In [12]:
# IMPORTAR ARCHIVOS
customers           = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/customers_dataset.csv')
geolocation_a       = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/geolocation_dataset_a.csv')
geolocation_b       = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/geolocation_dataset_b.csv')
geolocation_c       = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/geolocation_dataset_c.csv')
order_items         = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/order_items_dataset.csv')
order_payments      = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/order_payments_dataset.csv')
order_reviews       = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/order_reviews_dataset.csv')
orders              = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/orders_dataset.csv')
pc_name_translation = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/product_category_name_translation.csv')
products            = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/products_dataset.csv')
sellers             = pd.read_csv('https://raw.githubusercontent.com/kenzoyanome/brazilian_ecommerce/refs/heads/main/sellers_dataset.csv')

In [13]:
# CONCATENAMOS LOS GEOLOCATION FILES
geolocation         = pd.concat([geolocation_a, geolocation_b, geolocation_c], ignore_index=True)

In [18]:
# MOSTRAR HEADS
dfs = {'customers': customers,
       'geolocation': geolocation,
       'order_items': order_items,
       'order_payments': order_payments,
       'order_reviews': order_reviews,
       'orders': orders,
       'pc_name_translation': pc_name_translation,
       'products': products,
       'sellers': sellers
       }

def heads(dfs:dict):
  for name, df in dfs.items():
    print(f"Head de: {name}:")
    display(df.head())


heads(dfs)

Head de: customers:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


Head de: geolocation:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.546,-46.639,sao paulo,SP
1,1046,-23.546,-46.645,sao paulo,SP
2,1046,-23.546,-46.643,sao paulo,SP
3,1041,-23.544,-46.639,sao paulo,SP
4,1035,-23.542,-46.642,sao paulo,SP


Head de: order_items:


...

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


Head de: products:


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.000,287.000,1.000,225.000,16.000,10.000,14.000
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.000,276.000,1.000,1000.000,30.000,18.000,20.000
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.000,250.000,1.000,154.000,18.000,9.000,15.000
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.000,261.000,1.000,371.000,26.000,4.000,26.000
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.000,402.000,4.000,625.000,20.000,17.000,13.000


Head de: sellers:


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [19]:
# explorar datasets
def shapes(dfs:dict):
  for name, df in dfs.items():
    print(f"Shape de: {name}:")
    display(df.shape)


shapes(dfs)

Shape de: customers:


(99441, 5)

Shape de: geolocation:


(1000163, 5)

Shape de: order_items:


...

(71, 2)

Shape de: products:


(32951, 9)

Shape de: sellers:


(3095, 4)

In [22]:
customers.dtypes            # tipos de dato de cada columna
customers.isnull().sum()    # nulos por columna
customers.duplicated().sum()               # filas 100% repetidas
customers["customer_id"].is_unique         # ¿es clave primaria?
customers["customer_unique_id"].is_unique  # ¿es clave primaria?

customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
0
True
False
